# Purple AI V1.0 — trening od zera w Google Colab

Notebook pobiera 11 000 polskich dokumentów z otwartego korpusu Polish DynaWord, przygotowuje `training/data.txt`, uruchamia **nasz własny model z losowych wag** i na końcu automatycznie pobiera ZIP z checkpointem.

**Nie jest potrzebny token Hugging Face.** W Colab wybierz `Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU`, jeśli GPU jest dostępne.

Dane: `SlayerLab/polish-dynaword`, stabilna wersja `v0.2.5`, licencja korpusu CC-BY-SA-4.0. Notebook wybiera dokumenty z wybranych źródeł, zapisuje informację o źródle i licencji. Zobacz kartę zbioru przed dalszą dystrybucją danych lub wag.

In [ ]:
import os, sys, json, shutil, zipfile, subprocess
from pathlib import Path

REPO = Path('/content/PurpleAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kaielk09/Ai.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

%pip -q install datasets==3.3.2

print('Repo:', REPO)
print('PyTorch:', __import__('torch').__version__)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'brak — Colab może trenować wolno na CPU')

## 1. Pobierz około 11 000 polskich tekstów

To jest **pretraining językowy**: model uczy się przewidywać kolejne bajty tekstu. Sam zbiór dokumentów nie uczy jeszcze poprawnego formatu asystenta ani wykonywania poleceń.

In [ ]:
from datasets import load_dataset
from collections import Counter

TARGET_ROWS = 11_000
MAX_CHARS_PER_DOC = 5_000
ALLOWED_SOURCES = {
    'wikibooks', 'wikivoyage', 'wolne_lektury',
    'wiktionary_examples', 'global_voices', 'govpl',
    'samorzad_gov_pl', 'nkjp1m', 'wikiquote', 'wikinews'
}

stream = load_dataset(
    'SlayerLab/polish-dynaword',
    revision='v0.2.5',
    split='train',
    streaming=True,
).shuffle(seed=321, buffer_size=20_000)

texts = []
source_counts = Counter()
scanned = 0
for row in stream:
    scanned += 1
    source = str(row.get('source', '')).strip()
    text = row.get('text', '')
    if source not in ALLOWED_SOURCES or not isinstance(text, str):
        if scanned >= 500_000:
            break
        continue
    text = ' '.join(text.split()).strip()
    if len(text) < 100:
        continue
    texts.append(text[:MAX_CHARS_PER_DOC])
    source_counts[source] += 1
    if len(texts) >= TARGET_ROWS:
        break
    if scanned >= 500_000:
        break

if len(texts) < TARGET_ROWS:
    raise RuntimeError(f'Pobrano tylko {len(texts)} dokumentów po sprawdzeniu {scanned} rekordów. Nie uruchamiam treningu na niepełnym zbiorze.')

data_dir = REPO / 'training'
data_dir.mkdir(parents=True, exist_ok=True)
data_path = data_dir / 'data.txt'
data_path.write_text('\n\n'.join(texts), encoding='utf-8')

manifest = {
    'dataset': 'SlayerLab/polish-dynaword',
    'revision': 'v0.2.5',
    'dataset_license': 'CC-BY-SA-4.0',
    'rows': len(texts),
    'sources': dict(source_counts),
    'max_chars_per_document': MAX_CHARS_PER_DOC,
    'note': 'Training subset; retain attribution and comply with upstream per-source rights.'
}
(REPO / 'training_data_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Gotowe:', len(texts), 'dokumentów')
print('Źródła:', dict(source_counts))
print('Rozmiar data.txt:', round(data_path.stat().st_size / 1_000_000, 2), 'MB')

## 2. Ustawienia i trening

Checkpoint będzie zapisywany co 100 kroków. Jeśli Colab przerwie sesję, ponowne uruchomienie komórek może wznowić trening z ostatniego zapisu, o ile katalog `/content/purple_ai_output` nadal istnieje. Zapisuj kopię checkpointu, gdy trening trwa.

In [ ]:
OUTPUT = Path('/content/purple_ai_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['PURPLE_DATA'] = str(data_path)
os.environ['PURPLE_CHECKPOINT_DIR'] = str(OUTPUT)
os.environ['PURPLE_STEPS'] = '5000'
os.environ['PURPLE_BATCH_SIZE'] = '8'
os.environ['PURPLE_BLOCK_SIZE'] = '512'
os.environ['PURPLE_EVAL_EVERY'] = '100'
os.environ['PURPLE_LR'] = '0.0003'

train_script = REPO / 'training' / 'train_from_scratch.py'
subprocess.run([sys.executable, str(train_script)], cwd=str(REPO), check=True)

checkpoint = OUTPUT / 'purple_ai_latest.pt'
if not checkpoint.is_file():
    raise FileNotFoundError('Nie znaleziono checkpointu — trening nie zakończył się poprawnie.')
print('Checkpoint gotowy:', checkpoint, 'Rozmiar:', round(checkpoint.stat().st_size / 1_000_000, 2), 'MB')

## 3. Sprawdź i automatycznie pobierz wynik

Plik ZIP zawiera checkpoint `.pt`, konfigurację tokenizera, manifest danych i raport. **Nie wysyłaj pliku `training/data.txt` na publiczne GitHub** — zawiera pełne teksty treningowe.

In [ ]:
import torch

saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
report = {
    'model': 'Purple AI V1.0',
    'training_completed_steps': int(saved.get('step', 0)),
    'config': saved.get('config', {}),
    'last_training_loss': saved.get('loss'),
    'last_validation_loss': saved.get('validation_loss'),
    'dataset_rows': TARGET_ROWS,
    'dataset': 'SlayerLab/polish-dynaword@v0.2.5',
    'training_type': 'from_random_initialization'
}
(OUTPUT / 'training_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT / 'DATASET_ATTRIBUTION.txt').write_text(
    'Dataset: Polish DynaWord — SlayerLab/polish-dynaword, revision v0.2.5.\\n'
    'Dataset card: https://huggingface.co/datasets/SlayerLab/polish-dynaword\\n'
    'Corpus license: CC-BY-SA-4.0; underlying sources may have individual terms.\\n'
    'Keep this attribution and review the dataset card before redistributing derived artifacts.\\n',
    encoding='utf-8'
)

bundle = Path('/content/Purple_AI_V1_training_result.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(checkpoint, 'purple_ai_latest.pt')
    tokenizer = OUTPUT / 'tokenizer.json'
    if tokenizer.exists(): z.write(tokenizer, 'tokenizer.json')
    z.write(OUTPUT / 'training_report.json', 'training_report.json')
    z.write(OUTPUT / 'DATASET_ATTRIBUTION.txt', 'DATASET_ATTRIBUTION.txt')

print(json.dumps(report, ensure_ascii=False, indent=2))
from google.colab import files
files.download(str(bundle))

## Ważne po treningu

- To jest pierwszy mały etap pretrainingu, nie gotowy asystent konwersacyjny.
- `loss` i `validation_loss` pokazują postęp przewidywania tekstu, ale nie dowodzą poprawności odpowiedzi.
- Zbiór ma licencję CC-BY-SA-4.0 i dokumentację praw poszczególnych źródeł; zachowaj plik atrybucji.
- Jutro prześlij mi pobrany ZIP w rozmowie. Wtedy sprawdzimy checkpoint, konfigurację i zgodność z API, a następnie przygotujemy commit do repozytorium. Nie mam osobnego prywatnego magazynu, do którego mogę samodzielnie odłożyć plik poza tą rozmową.